# LINEBENCH — Notebook 05: Simulator Validation, Timing, and Convergence

**Phase 5 Deliverable**

This notebook demonstrates and validates the simulation engines developed in Phase 5:
1. **Noise Generator (`noise.py`)**: Lognormal & Gamma variability with Common Random Numbers (CRN).
2. **Station-Time Builder (`stations.py`)**: Fast matrix multiplication assignment mapping.
3. **Unpaced Tandem Line (`tandem.py`)**: Dynamic programming departure-time recursion with blocking-after-service (BAS).
4. **Paced Model (`paced.py`)**: Wait-for-slowest synchronous line and analytical normal approximations.
5. **SimPy Discrete-Event Validation (`simpy_line.py`)**: Structural proof of identical departure times.
6. **Analytical 2-Station Queueing Validation**: $M/M/1/b$ tandem queue agreement against $\mu \frac{b+2}{b+3}$.
7. **Timing Benchmark & Convergence Study (`convergence.py`)**: Vectorized speedup and precision curves.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore', 'FigureCanvasAgg is non-interactive')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# Ensure project root and src are in path regardless of notebook working directory
project_root = Path.cwd()
if not (project_root / 'src').exists() and (project_root.parent / 'src').exists():
    project_root = project_root.parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from linebench.simulation.convergence import run_timing_benchmark, run_convergence_study
from linebench.simulation.noise import generate_noise_factors
from linebench.simulation.paced import simulate_paced_line, normal_approx_overload_probability
from linebench.simulation.simpy_line import simulate_simpy_line
from linebench.simulation.stations import assignment_to_matrix, build_station_times
from linebench.simulation.tandem import simulate_tandem_line, simulate_tandem_replications

print(f'All simulation modules successfully imported. Project root: {project_root}')

## 1. Noise Generator & Common Random Numbers (CRN)

Verifying that the multiplicative noise generator produces mean-one distributions with exact target CVs.

In [ ]:
n_samples = 50000
cv_target = 0.25
ln_factors = generate_noise_factors((n_samples,), cv=cv_target, seed=42, distribution='lognormal')
ga_factors = generate_noise_factors((n_samples,), cv=cv_target, seed=42, distribution='gamma')

print(f'Lognormal (target CV={cv_target}): Mean = {ln_factors.mean():.4f}, CV = {ln_factors.std()/ln_factors.mean():.4f}')
print(f'Gamma     (target CV={cv_target}): Mean = {ga_factors.mean():.4f}, CV = {ga_factors.std()/ga_factors.mean():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ln_factors, bins=60, density=True, alpha=0.6, label='Lognormal (primary)', color='steelblue')
ax.hist(ga_factors, bins=60, density=True, alpha=0.5, label='Gamma (ablation)', color='darkorange')
ax.axvline(1.0, color='red', linestyle='--', label='Theoretical Mean E[X]=1.0')
ax.set_title(f'Simulated Noise Distributions (CV = {cv_target})')
ax.set_xlabel('Multiplicative Noise Factor')
ax.set_ylabel('Probability Density')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Event-Driven SimPy vs Dynamic Recursion

Feeding the exact same stochastic service times into both the NumPy recursion and SimPy.
The departure times must match to floating-point precision.

In [ ]:
rng = np.random.default_rng(2026)
N_units = 100
K_stations = 4
buffer_cap = 1
S_sample = rng.lognormal(mean=2.0, sigma=0.3, size=(N_units, K_stations))

s_simpy, f_simpy, d_simpy = simulate_simpy_line(S_sample, buffers=buffer_cap)
res_rec = simulate_tandem_line(S_sample, buffers=buffer_cap, warmup_units=0, return_details=True)

max_diff = np.max(np.abs(res_rec.departure_times - d_simpy))
print(f'SimPy vs Recursion Max Departure Difference: {max_diff:.2e}')
assert max_diff < 1e-9, 'Departure times differ!'

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(res_rec.departure_times[:25, -1], label='Dynamic Recursion Departures', marker='o', linestyle='-', color='teal')
ax.plot(d_simpy[:25, -1], label='SimPy Discrete-Event Departures', marker='x', linestyle='--', color='crimson')
ax.set_title('First 25 Units: Exit Departures from Final Station (Recursion vs SimPy)')
ax.set_xlabel('Unit Index (i)')
ax.set_ylabel('Departure Time (d_K,i)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Analytical Validation: Two-Station Markov Tandem Queue

Testing the unpaced recursion against the closed-form theoretical formula:
$$\text{Throughput}(b, \mu) = \mu \frac{b + 2}{b + 3}$$
for $b \in \{0, 1, 2\}$.

In [ ]:
mu = 1.0
reps = 30
N_sim = 8000
W_sim = 1500

results_analytical = []
for b in [0, 1, 2]:
    theory_th = mu * (b + 2) / (b + 3)
    ths = []
    for r in range(reps):
        S_exp = rng.exponential(scale=1.0/mu, size=(N_sim, 2))
        res = simulate_tandem_line(S_exp, buffers=b, warmup_units=W_sim)
        ths.append(res.throughput)
    
    mean_emp = np.mean(ths)
    ci_half = stats.t.ppf(0.995, df=reps-1) * stats.sem(ths)
    results_analytical.append({
        'buffer_b': b,
        'theoretical_throughput': theory_th,
        'simulated_mean': mean_emp,
        'ci_99_lower': mean_emp - ci_half,
        'ci_99_upper': mean_emp + ci_half,
        'matches_theory_ci99': (mean_emp - ci_half) <= theory_th <= (mean_emp + ci_half)
    })

df_analytical = pd.DataFrame(results_analytical)
print(df_analytical.to_string(index=False))

## 4. Paced Line Model & Analytical Normal Approximation

Demonstrating wait-for-slowest paced simulation and comparing empirical overload probabilities with the normal approximation.

In [ ]:
tasks = [8.0, 12.0, 10.0, 14.0, 6.0]  # sum = 50.0
c_nominal = 55.0
cv_level = 0.25

norm_p = normal_approx_overload_probability(tasks, cycle_time=c_nominal, cv=cv_level)

# Empirical simulation
A_dummy = {1: list(range(1, len(tasks) + 1))}
noise = generate_noise_factors((1, 5000, len(tasks)), cv=cv_level, seed=42)
S_paced = build_station_times(A_dummy, tasks, noise)[0]  # (5000, 1)
paced_res = simulate_paced_line(S_paced, cycle_time=c_nominal)

print(f'Analytical Normal Approx Overload P(S > {c_nominal}): {norm_p:.4f}')
print(f'Empirical Simulation Overload P(S > {c_nominal}):       {paced_res.station_overload_prob[0]:.4f}')

## 5. Execution Speed & Convergence Results

Loading the benchmark and convergence datasets generated in Phase 5.7 and 5.8.

In [ ]:
results_dir = project_root / 'results'
df_timing = pd.read_csv(results_dir / 'simulation_timing_benchmark.csv')
print('=== Simulation Timing Benchmark ===')
print(df_timing.to_string(index=False))

df_conv = pd.read_csv(results_dir / 'simulation_convergence_study.csv')
print('\n=== Convergence at Protocol Settings (W=500, N=5500, reps=30) ===')
target_row = df_conv[(df_conv['warmup_units'] == 500) & (df_conv['total_units'] == 5500)]
print(target_row[['warmup_units', 'total_units', 'n_replications', 'mean_throughput', 'sem_throughput', 'ci_95_half_width', 'relative_ci_width_pct']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sub_500 = df_conv[df_conv['warmup_units'] == 500].sort_values('total_units')
ax.plot(sub_500['total_units'], sub_500['relative_ci_width_pct'], marker='o', color='purple', lw=2)
ax.axhline(0.1, color='red', linestyle='--', label='0.1% Relative Error Threshold')
ax.set_title('Simulation Precision vs Total Units (Warmup = 500)')
ax.set_xlabel('Total Units (N)')
ax.set_ylabel('95% CI Half-Width (% of Mean)')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()